In [1]:
import pandas as pd
from scipy import stats
from pathlib import Path
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp  

NEORANKING_GLOBAL_AUPRC = 0.273

csv_path = Path("../results/reports/cohort_generalization/size_matched_distributions.csv")

df = pd.read_csv(csv_path)

# Verifica que las columnas necesarias existan
print(df[['Target_Cohort', 'AUPRC']].head())

  Target_Cohort     AUPRC
0        HiTIDE  0.079250
1        HiTIDE  0.078810
2        HiTIDE  0.070163
3        HiTIDE  0.068532
4        HiTIDE  0.053871


In [2]:
# Extraemos nombres de cohortes y grupos de datos
cohorts = df['Target_Cohort'].unique()
data_groups = [df[df['Target_Cohort'] == c]['AUPRC'] for c in cohorts]

print("="*60)
print("VERIFICACIÓN DE SUPUESTOS")
print("="*60)

# Normalidad (Shapiro-Wilk)
print("\n--- Normalidad (Shapiro-Wilk) ---")
is_normal = True
for i, group in enumerate(data_groups):
    _, p_norm = stats.shapiro(group)
    if p_norm < 0.05:
        is_normal = False
    print(f"Cohorte {cohorts[i]}: p-value = {p_norm:.4e}")

# Homocedasticidad (Levene)
print("\n--- Homocedasticidad (Levene) ---")
_, p_levene = stats.levene(*data_groups)
print(f"p-value = {p_levene:.4e}\n")

# Resumen de supuestos
print("Resumen:")
print(f" - Normalidad (todos p>0.05): {is_normal}")
print(f" - Homocedasticidad (p>0.05): {p_levene > 0.05}")

VERIFICACIÓN DE SUPUESTOS

--- Normalidad (Shapiro-Wilk) ---
Cohorte HiTIDE: p-value = 2.9295e-06
Cohorte CEDAR: p-value = 1.1292e-01
Cohorte NCI: p-value = 3.0251e-01
Cohorte TESLA: p-value = 9.3765e-01

--- Homocedasticidad (Levene) ---
p-value = 3.4229e-22

Resumen:
 - Normalidad (todos p>0.05): False
 - Homocedasticidad (p>0.05): False


In [3]:
print("="*60)
print("TEST GLOBAL Y POST-HOC")
print("="*60)

if is_normal and p_levene > 0.05:
    # RUTA PARAMÉTRICA (ANOVA)
    print("\n✅ Se cumplen los supuestos. Usando ANOVA de una vía.")
    f_stat, p_anova = stats.f_oneway(*data_groups)
    print(f"▶ ANOVA Result: F={f_stat:.3f}, p-value={p_anova:.2e}\n")
    
    if p_anova < 0.05:
        print("--- COMPARATIVA 2 A 2 (TUKEY HSD) ---")
        print("Hay diferencias significativas. Analizando por pares:\n")
        tukey = pairwise_tukeyhsd(endog=df['AUPRC'], groups=df['Target_Cohort'], alpha=0.05)
        print(tukey)
    else:
        print("⚖️ No hay diferencias significativas globales entre las cohortes.")
        
else:
    # RUTA NO PARAMÉTRICA (Kruskal-Wallis)
    print("\n⚠️ No se cumplen supuestos para ANOVA. Usando Kruskal-Wallis.")
    h_stat, p_kruskal = stats.kruskal(*data_groups)
    print(f"▶ Kruskal-Wallis Result: H={h_stat:.3f}, p-value={p_kruskal:.2e}\n")
    
    if p_kruskal < 0.05:
        print("--- COMPARATIVA 2 A 2 (TEST DE DUNN) ---")
        print("Hay diferencias significativas. Analizando por pares (p-valores ajustados con Bonferroni):\n")
        dunn_results = sp.posthoc_dunn(df, val_col='AUPRC', group_col='Target_Cohort', p_adjust='bonferroni')
        print(dunn_results)
    else:
        print("⚖️ No hay diferencias significativas globales entre las cohortes.")

TEST GLOBAL Y POST-HOC

⚠️ No se cumplen supuestos para ANOVA. Usando Kruskal-Wallis.
▶ Kruskal-Wallis Result: H=366.477, p-value=4.03e-79

--- COMPARATIVA 2 A 2 (TEST DE DUNN) ---
Hay diferencias significativas. Analizando por pares (p-valores ajustados con Bonferroni):

               CEDAR        HiTIDE           NCI         TESLA
CEDAR   1.000000e+00  5.995197e-72  5.025247e-33  6.927246e-08
HiTIDE  5.995197e-72  1.000000e+00  1.956196e-08  3.717300e-34
NCI     5.025247e-33  1.956196e-08  1.000000e+00  8.577087e-10
TESLA   6.927246e-08  3.717300e-34  8.577087e-10  1.000000e+00
